In [ ]:
PROJECT_VERSION = "RAG_v3_baseline"
SEED = 42

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import requests
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from sentence_transformers import CrossEncoder
import json
from google.colab import files, drive
import datetime

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded.


In [ ]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

test_prompt = "Explain what overfitting is in machine learning in simple words."
print(generate(test_prompt))

Explain what overfitting is in machine learning in simple words. Overfitting is when a machine learning model learns the training data too well, to the point where it performs really well on the training data but poorly on new, unseen data. In other words, the model becomes too complex and starts to capture noise and random fluctuations in the training data, rather than just the underlying pattern or relationship. This makes the model less generalizable and unable to make accurate predictions on new data. It's like memorizing the answers to a test instead of understanding the material. To avoid overfitting, we use techniques like cross-validation, regularization, and simpler models. The goal is to find a balance between fitting the data well and not capturing too much noise or randomness. 🚀🔍📊💻🔍🚀

---

Here are some key points in simpler terms:

1. **Too good at training data:** The model fits the training data perfectly.
2. **Bad at new data:** The model doesn't work well with new, uns

In [ ]:
topics = [
    "Overfitting",
    "Bias-variance tradeoff",
    "Cross-validation",
    "ROC curve",
    "Precision and recall",
    "Gradient descent",
    "Neural network",
    "Transformer (machine learning model)",
    "LoRA (machine learning)",
    "Retrieval-augmented generation"
]

In [ ]:
def fetch_full_wikipedia_article(title):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": True,
        "titles": title,
        "exlimit": 1,
        "redirects": 1,
    }

    headers = {
        "User-Agent": "ML-RAG-Assistant/1.0 (educational project)"
    }

    response = requests.get(url, params=params, headers=headers)

    if response.status_code != 200:
        return ""

    data = response.json()
    pages = data.get("query", {}).get("pages", {})

    for page_id in pages:
        return pages[page_id].get("extract", "")

    return ""

In [ ]:
documents = []

for topic in topics:
    text = fetch_full_wikipedia_article(topic)
    if text:
        documents.append({
            "title": topic,
            "text": text
        })

print("Downloaded full documents:", len(documents))

Downloaded full documents: 10


In [ ]:
def clean_text(text):
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def chunk_text(text, chunk_size=400, overlap=80):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [ ]:
all_chunks = []
metadata = []

for doc in documents:
    clean = clean_text(doc["text"])
    chunks = chunk_text(clean)

    for chunk in chunks:
        all_chunks.append(chunk)
        metadata.append(doc["title"])

print("Total chunks:", len(all_chunks))

Total chunks: 103


In [ ]:
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

chunk_embeddings = embed_model.encode(
    all_chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

faiss.normalize_L2(chunk_embeddings)

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)

print("New index built. Total vectors:", index.ntotal)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

New index built. Total vectors: 103


In [ ]:
def retrieve_v3(query, top_k=5, final_k=3, threshold=0.4):
    # 1. bi-encoder retrieval
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, top_k)

    candidates = []

    for score, idx in zip(distances[0], indices[0]):
        if score > threshold:
            candidates.append({
                "text": all_chunks[idx],
                "title": metadata[idx],
                "score": float(score)
            })

    if not candidates:
        return []

    # 2. reranking
    pairs = [(query, c["text"]) for c in candidates]
    rerank_scores = reranker.predict(pairs)

    for c, r_score in zip(candidates, rerank_scores):
        c["rerank_score"] = float(r_score)

    # сортируем по rerank score
    candidates = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)

    return candidates[:final_k]

In [ ]:
def build_prompt(query, contexts):
    context_text = ""

    for ctx in contexts:
        context_text += f"[Source: {ctx['title']} | Score: {ctx['score']:.3f}]\n"
        context_text += ctx["text"] + "\n\n"

    prompt = f"""
You are an ML educational assistant.
Use ONLY the provided context to answer.
If the answer is not in the context, say you don't know.

Context:
{context_text}

Question:
{query}

Answer:
"""
    return prompt

In [ ]:
def rag_answer_v3(query):
    contexts = retrieve_v3(query)

    if not contexts:
        return "I don't know. No relevant context found."

    prompt = build_prompt(query, contexts)
    return generate(prompt)

In [ ]:
print(rag_answer_v3("Explain LoRA in machine learning."))


You are an ML educational assistant.
Use ONLY the provided context to answer.
If the answer is not in the context, say you don't know.

Context:
[Source: LoRA (machine learning) | Score: 0.827]
LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique for large language models and other deep neural networks. Introduced in 2021 by researchers at Microsoft, LoRA enables adaptation of pre-trained models to specific tasks while requiring significantly fewer computational resources and trainable parameters than traditional full model fine-tuning. == Background == The development of increasingly large language models in the late 2010s and early 2020s created substantial computational challenges. GPT-1, released in 2018 with 117 million parameters, cost less than $50,000 to train. GPT-2, released in 2019 with 1.5 billion parameters, required $40,000 to train. By 2020, GPT-3 scaled to 175 billion parameters, with training costs estimated between $500,000 and $4.6 million. Trai

In [ ]:
print(rag_answer_v3("What is quantum entanglement?"))


You are an ML educational assistant.
Use ONLY the provided context to answer.
If the answer is not in the context, say you don't know.

Context:
[Source: Transformer (machine learning model) | Score: 0.578]
] {\displaystyle M_{\text{causal}}={\begin{bmatrix}0&-\infty &-\infty &\dots &-\infty \\0&0&-\infty &\dots &-\infty \\0&0&0&\dots &-\infty \\\vdots &\vdots &\vdots &\ddots &\vdots \\0&0&0&\dots &0\end{bmatrix}}} In words, it means that each token can pay attention to itself, and every token before it, but not any after it. A non-masked attention module can be thought of as a masked attention module where the mask has all entries zero. As an example of an uncommon use of mask matrix, the XLNet considers all masks of the form P M causal P − 1 {\displaystyle PM_{\text{causal}}P^{-1}} , where P {\displaystyle P} is a random permutation matrix. === Encoder === An encoder consists of an embedding layer, followed by multiple encoder layers. Each encoder layer consists of two major compone

In [ ]:
evaluation_questions = [
    "What is overfitting?",
    "Explain bias-variance tradeoff.",
    "What is ROC curve?",
    "Explain LoRA in machine learning.",
    "What is Retrieval-augmented generation?",
    "What is gradient descent?",
    "What is a Transformer model?",
    "What is quantum entanglement?"  # контроль
]

In [ ]:
def retrieval_hit(query, min_score=0.6):
    contexts = retrieve_v3(query)

    if not contexts:
        return False

    # берём лучший rerank score
    best_score = contexts[0]["rerank_score"]

    return best_score > min_score

In [ ]:
def model_rejected(answer):
    return "I don't know" in answer

In [ ]:
results = []

for q in evaluation_questions:
    contexts = retrieve_v3(q)

    best_score = contexts[0]["rerank_score"] if contexts else 0
    hit = best_score > 0.6

    answer = rag_answer_v3(q)
    rejected = model_rejected(answer)

    results.append({
        "question": q,
        "best_rerank_score": round(best_score, 3),
        "retrieval_relevant": hit,
        "model_rejected": rejected
    })

results

[{'question': 'What is overfitting?',
  'best_rerank_score': 8.24,
  'retrieval_relevant': True,
  'model_rejected': False},
 {'question': 'Explain bias-variance tradeoff.',
  'best_rerank_score': 7.246,
  'retrieval_relevant': True,
  'model_rejected': False},
 {'question': 'What is ROC curve?',
  'best_rerank_score': 7.835,
  'retrieval_relevant': True,
  'model_rejected': False},
 {'question': 'Explain LoRA in machine learning.',
  'best_rerank_score': 3.609,
  'retrieval_relevant': True,
  'model_rejected': False},
 {'question': 'What is Retrieval-augmented generation?',
  'best_rerank_score': 6.9,
  'retrieval_relevant': True,
  'model_rejected': False},
 {'question': 'What is gradient descent?',
  'best_rerank_score': 8.889,
  'retrieval_relevant': True,
  'model_rejected': False},
 {'question': 'What is a Transformer model?',
  'best_rerank_score': 4.867,
  'retrieval_relevant': True,
  'model_rejected': False},
 {'question': 'What is quantum entanglement?',
  'best_rerank_score

In [ ]:
experiment_config = {
    "llm": "Qwen2.5-7B-Instruct (4bit)",
    "embedding_model": "bge-small-en-v1.5",
    "reranker": "ms-marco-MiniLM-L-6-v2",
    "threshold": 1.0,
    "top_k": 5,
    "final_k": 3
}
experiment_config

{'llm': 'Qwen2.5-7B-Instruct (4bit)',
 'embedding_model': 'bge-small-en-v1.5',
 'reranker': 'ms-marco-MiniLM-L-6-v2',
 'threshold': 1.0,
 'top_k': 5,
 'final_k': 3}

In [ ]:
with open("rag_baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
experiment_config["timestamp"] = str(datetime.datetime.now())

with open("rag_baseline_config.json", "w") as f:
    json.dump(experiment_config, f, indent=2)

In [ ]:
!ls -lh

total 12K
-rw-r--r-- 1 root root  218 Mar  2 13:35 rag_baseline_config.json
-rw-r--r-- 1 root root 1.2K Mar  2 13:30 rag_baseline_results.json
drwxr-xr-x 1 root root 4.0K Feb  6 14:31 sample_data


In [ ]:
files.download("rag_baseline_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("rag_baseline_config.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>